# New Mexico Water Use Analysis - Agriculture

Extract county-level agriculture water use from NM OSE XLSX of 2020.
Returns tidy DataFrame: county_key, year, tpwsw_af, tpwgw_af, tpw_af


TPWSW — Total Produced Water, Surface Water. Irrigation water withdrawn from rivers/streams (acre-feet).

TPWGW — Total Produced Water, Groundwater. Irrigation water pumped from aquifers (acre-feet). Often larger in NM due to aridity.

TPW — Total Produced Water = TPWSW + TPWGW. Primary demand metric.

TFWSW — Total Freshwater Surface Water for crop production only. (ignore: it's a subset, not the full demand)

In [ ]:
!pip install openpyxl xlrd -q
import pandas as pd
import numpy as np
import geopandas as gpd
import os, requests

N_COLAB = 'google.colab' in str(dir())
if IN_COLAB:
    from google.colab import drive
    drive.mount('/content/drive')
    WORK_DIR = "/content/drive/MyDrive/YOUR_FOLDER/pw_analysis"
else:
    WORK_DIR = "."

DATA_DIR = f"{WORK_DIR}/data_processed"
RAW_DIR  = f"{WORK_DIR}/data_raw/"

FILE_2020 = f"{RAW_DIR}/demand/wateruse2020.xlsx"
FILE_2015 = f"{RAW_DIR}/demand/wateruse2015.xlsx"


for f in [FILE_2020, FILE_2015]:
    exists = os.path.exists(f)
    print(f"{'✓' if exists else '✗'} {f.split('/')[-1]}")

In [ ]:
# Print every sheet name in both files
for label, fpath in [("2020", FILE_2020), ("2015", FILE_2015)]:
    xl = pd.ExcelFile(fpath)
    print(f"\n=== {label} file — sheets ===")
    for name in xl.sheet_names:
        print(f"  '{name}'")

In [ ]:
# Table 8 from the 2020 file

t8 = pd.read_excel(FILE_2020, sheet_name="8. IrrigatedAg", header=None)
print("=== Table 8 raw (first 10 rows) ===")
print(t8.head(10).to_string()) # print

# Table 9
t9 = pd.read_excel(FILE_2020, sheet_name="9. AcresCounty", header=None)
print("\n=== Table 9 raw (first 10 rows) ===")
print(t9.head(10).to_string()) # print

In [ ]:
for label, fpath in [("2020", FILE_2020)]:
    print(f"\n=== {label} — Table 1 (population info) ===")
    t1 = pd.read_excel(fpath, sheet_name="1. Pop", header=3)
    print(t1.head(15).to_string())

    print(f"\n=== {label} — Table 2 (WSW, WGW, TW by sector) ===")
    t2 = pd.read_excel(fpath, sheet_name="8. IrrigatedAg", header=3)
    print(t2.head(15).to_string())

In [ ]:
def extract_agriculture(filepath, year, sheet="8. IrrigatedAg", header_row=2):
    """
    Extract county-level agriculture water use from NM OSE XLSX.
    Returns tidy DataFrame: county_key | year | tpwsw_af | tpwgw_af | tpw_af
    """
    # Read the sheet with the specified header row
    df = pd.read_excel(filepath, sheet_name=sheet, header=header_row)

    # Normalise column names to upper case and strip whitespace
    df.columns = [str(c).strip().upper() for c in df.columns]

    print(f"  Columns found in sheet '{sheet}': {df.columns.tolist()}")
    print(f"  DF head after initial read and column normalization:\n{df.head().to_string()}") # Debug print

    # Identify the county column. It should be 'COUNTY' after normalization.
    if 'COUNTY' not in df.columns:
        print("ERROR: 'COUNTY' column not found. Check header_row and sheet content.")
        return pd.DataFrame()
    county_col = 'COUNTY'
    print(f"  County column: '{county_col}'") # Completed the print statement

    # Select relevant columns for water use
    keep_cols_data = {
        "TPWSW": "tpwsw_af",
        "TPWGW": "tpwgw_af"
    }

    selected_cols = [county_col]
    renamed_cols = ["county_raw"]

    for original_col, new_name in keep_cols_data.items():
        if original_col in df.columns:
            selected_cols.append(original_col)
            renamed_cols.append(new_name)
            print(f"  Found {original_col} → '{new_name}'")
        else:
            print(f"  WARNING: {original_col} not found in columns")

    if len(selected_cols) == 1: # Only county_col was found, no water data
        print("ERROR: No water columns found. Returning empty DataFrame.")
        return pd.DataFrame()

    df_clean = df[selected_cols].copy()
    df_clean.columns = renamed_cols

    # Calculate Total Public Water (tpw_af) if both components are available
    if 'tpwsw_af' in df_clean.columns and 'tpwgw_af' in df_clean.columns:
        df_clean['tpw_af'] = pd.to_numeric(df_clean['tpwsw_af'], errors='coerce').fillna(0) + \
                             pd.to_numeric(df_clean['tpwgw_af'], errors='coerce').fillna(0)
    elif 'tpwsw_af' in df_clean.columns:
        df_clean['tpw_af'] = pd.to_numeric(df_clean['tpwsw_af'], errors='coerce').fillna(0)
    elif 'tpwgw_af' in df_clean.columns:
        df_clean['tpw_af'] = pd.to_numeric(df_clean['tpwgw_af'], errors='coerce').fillna(0)
    else:
        df_clean['tpw_af'] = 0 # No water columns found

    # Drop header repeats, totals, and empty rows
    # Convert to uppercase and replace spaces with underscores to match power data
    df_clean["county_key"] = df_clean["county_raw"].astype(str).str.strip().str.upper().str.replace(' ', '_')
    skip_values = ["COUNTY", "TOTAL", "NEW MEXICO", "NAN", "STATE", "SUM", ""]

    # Filter out rows where county_key is in skip_values or is NaN
    df_clean = df_clean[~df_clean["county_key"].isin(skip_values)]
    df_clean = df_clean.dropna(subset=["county_key"]) # Also drop if county_key is actual NaN

    # Convert to numeric, coercing errors to NaN and then filling with 0
    for col in [c for c in df_clean.columns if c not in ["county_raw", "county_key", "tpw_af"]]: # Exclude tpw_af as it was just calculated
        df_clean[col] = pd.to_numeric(df_clean[col], errors="coerce").fillna(0)

    df_clean["year"] = year
    df_clean = df_clean.drop(columns=["county_raw"])

    return df_clean.reset_index(drop=True)

# Extract from 2020 — try Table 8 first
print("=== Extracting 2020 agriculture data ===")
agri_2020 = extract_agriculture(FILE_2020, 2020, sheet="8. IrrigatedAg", header_row=2)
print(f"\nRows: {len(agri_2020)}")
print(agri_2020.head(10).to_string(index=False))

# If Table 8 is empty or wrong, try Table 9:
# agri_2020 = extract_agriculture(FILE_2020, 2020, sheet="Table 9")

In [ ]:
# Sanity check: do TPWSW + TPWGW = TPW for each row

agri_2020["tpw_check"] = agri_2020["tpwsw_af"] + agri_2020["tpwgw_af"]
agri_2020["tpw_diff"]  = (agri_2020["tpw_af"] - agri_2020["tpw_check"]).abs()

print("TPW reconciliation (should be close to 0):")
print(agri_2020[["county_key","tpwsw_af","tpwgw_af","tpw_af","tpw_diff"]]
      .sort_values("tpw_af", ascending=False).head(10).to_string(index=False))
print(f"\nMax discrepancy: {agri_2020['tpw_diff'].max():.1f} acre-ft")
print(f"Total NM irrigation demand 2020: {agri_2020['tpw_af'].sum():,.0f} acre-ft")

agri_2020 = agri_2020.drop(columns=["tpw_check","tpw_diff"])

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Sort the data by tpw_af for better visualization
agri_2020_sorted = agri_2020.sort_values(by='tpw_af', ascending=False)

plt.figure(figsize=(18, 10)) # Increased figure width and height for better label display
sns.barplot(x='county_key', y='tpw_af', data=agri_2020_sorted, palette='viridis', hue='county_key', legend=False)
plt.xlabel('County')
plt.ylabel('Total Public Water (acre-feet)')
plt.title('Total Public Water (tpw_af) by County in 2020')
plt.xticks(rotation=90)
plt.subplots_adjust(bottom=0.35) # Further increased bottom margin to 0.35
#plt.tight_layout()
plt.show()

Power Plant

In [ ]:
power_raw = pd.read_excel(FILE_2020, sheet_name="Power", header=None)
print("=== Power sheet (first 20 rows) ===")
print(power_raw.head(20).to_string())

# Find header row
for i, row in power_raw.iterrows():
    vals = [str(v).upper().strip() for v in row.values]
    if any(w in vals for w in ["PLANT","FACILITY","NAME","WSW","WGW","TW"]):
        print(f"\nLikely header at row {i}: {vals}")
        break

In [ ]:
def extract_power(filepath, year, sheet="Power", header_row=None):
    raw = pd.read_excel(filepath, sheet_name=sheet, header=None)

    if header_row is None:
        for i, row in raw.iterrows():
            vals = [str(v).upper().strip() for v in row.values]
            if any(w in vals for w in ["WSW","WGW","TW","PLANT","FACILITY"]):
                header_row = i
                print(f"  Header at row {i}: {vals}")
                break

    df = pd.read_excel(filepath, sheet_name=sheet, header=header_row)
    df.columns = [str(c).strip().upper() for c in df.columns]
    print(f"  Power columns: {df.columns.tolist()}")

    # Identify relevant columns
    name_col = next((c for c in df.columns if c == "NAME OF WATER USER" or c in ["PLANT","FACILITY"] ), None)
    if not name_col: # Fallback to first column if specific names not found
        name_col = df.columns[0]

    wsw_col = next((c for c in df.columns if "WSW" in c), None)
    wgw_col = next((c for c in df.columns if "WGW" in c), None)
    # Removed search for 'TW' column as it's not explicitly present and will be calculated
    county_col = next((c for c in df.columns if c == "COUNTY"), None)

    print(f"  Name col: {name_col}, WSW: {wsw_col}, WGW: {wgw_col}, County: {county_col}")

    # Select and rename columns
    selected_cols_temp = []
    new_names_temp = []

    if name_col:
        selected_cols_temp.append(name_col)
        new_names_temp.append("plant_name")

    # Handle WSW
    if wsw_col:
        selected_cols_temp.append(wsw_col)
        new_names_temp.append("wsw_af")
    else:
        # If wsw_col is not found, we need to add a placeholder column
        df["wsw_af_temp_col"] = 0.0 # Temporary column in df
        selected_cols_temp.append("wsw_af_temp_col")
        new_names_temp.append("wsw_af")

    # Handle WGW
    if wgw_col:
        selected_cols_temp.append(wgw_col)
        new_names_temp.append("wgw_af")
    else:
        # If wgw_col is not found, we need to add a placeholder column
        df["wgw_af_temp_col"] = 0.0 # Temporary column in df
        selected_cols_temp.append("wgw_af_temp_col")
        new_names_temp.append("wgw_af")

    if county_col:
        selected_cols_temp.append(county_col)
        new_names_temp.append("county")

    df_clean = df[selected_cols_temp].copy()
    df_clean.columns = new_names_temp

    df_clean["plant_name"] = df_clean["plant_name"].astype(str).str.strip()
    # Added 'state total' to skip list to exclude the aggregate row
    skip = ["plant","facility","name","total","nan","state total","", None]
    df_clean = df_clean[~df_clean["plant_name"].str.lower().isin(skip)]
    df_clean = df_clean.dropna(subset=["plant_name"])

    for col in [c for c in df_clean.columns if c not in ["plant_name", "county"]]:
        df_clean[col] = pd.to_numeric(df_clean[col], errors="coerce").fillna(0)

    # Calculate tw_af as sum of wsw_af and wgw_af
    df_clean["tw_af"] = df_clean["wsw_af"] + df_clean["wgw_af"]

    df_clean["year"] = year
    return df_clean.reset_index(drop=True)

print("=== Power plants 2020 ===")
power_2020 = extract_power(FILE_2020, 2020)
print(f"\nPlants: {len(power_2020)}")
print(power_2020.to_string(index=False))

In [ ]:
power_nm = power_2020 # Assuming power_2020 already contains data for New Mexico or is the intended base for aggregation.
power_county = (
    power_nm
    .groupby("county")
    .agg(
        plant_count  = ("plant_name", "count"),
        wsw_af_2020  = ("wsw_af",     "sum"),
        wgw_af_2020  = ("wgw_af",     "sum"),
        tw_af_2020   = ("tw_af",      "sum"),
    )
    .reset_index()
)
print("\nPower water use by county (2020):")
print(power_county)

### Save Processed Data

Saving the `agri_2020` DataFrame to a CSV file for future use.

In [ ]:
import os


os.makedirs(DATA_DIR, exist_ok=True)

output_file_path = f'{DATA_DIR}/agri_2020_processed.csv'
agri_2020.to_csv(output_file_path, index=False)

print(f"'agri_2020' DataFrame saved to: {output_file_path}")

In [ ]:
# Conversion constants
AF_TO_M3  = 1_233.48       # 1 acre-foot = 1,233.48 m³
AF_TO_BBL = 7_758.37       # 1 acre-foot = 7,758.37 barrels
M3_TO_BBL = AF_TO_BBL / AF_TO_M3   # = 6.2898

def convert_af_columns(df, af_cols):
    """
    Add m3 and bbl columns for each acre-foot column.
    Input:  df with columns named *_af
    Output: df with additional *_m3 and *_bbl columns added
    """
    df = df.copy()
    for col in af_cols:
        base = col.replace("_af","")
        df[f"{base}_m3"]  = (df[col] * AF_TO_M3).round(1)
        df[f"{base}_bbl"] = (df[col] * AF_TO_BBL).round(1)
    return df

# Apply to agriculture
agri_2020_conv = convert_af_columns(
    agri_2020,
    af_cols=["tpwsw_af","tpwgw_af","tpw_af"]
)

print("Agriculture 2020 — all units:")
print(agri_2020_conv[[
    "county_key","tpw_af","tpw_m3","tpw_bbl"
]].sort_values("tpw_bbl", ascending=False).head(10).to_string(index=False))

print(f"\nTotal NM irrigation demand 2020:")
print(f"  {agri_2020['tpw_af'].sum():>15,.0f}  acre-feet")
print(f"  {agri_2020['tpw_af'].sum()*AF_TO_M3:>15,.0f}  m³")
print(f"  {agri_2020['tpw_af'].sum()*AF_TO_BBL:>15,.0f}  bbl")

In [ ]:
def convert_af_columns(df, af_cols):
    df_conv = df.copy()
    CONVERSION_AF_TO_M3 = 1233.48
    CONVERSION_AF_TO_BBL = 7758 # Using 7758 as seen in other notebooks

    for col in af_cols:
        if col in df_conv.columns:
            # Convert acre-feet to cubic meters
            df_conv[col.replace("_af", "_m3")] = df_conv[col] * CONVERSION_AF_TO_M3
            # Convert acre-feet to barrels
            df_conv[col.replace("_af", "_bbl")] = df_conv[col] * CONVERSION_AF_TO_BBL

    # Add county_key if 'county' column exists
    if 'county' in df_conv.columns:
        df_conv['county_key'] = df_conv['county'].str.upper().str.replace(' ', '_')

    return df_conv

power_county_conv = convert_af_columns(
    power_county,
    af_cols=["wsw_af_2020","wgw_af_2020","tw_af_2020"]
)

print("Power plant water use by county — all units:")
print(power_county_conv[[
    "county_key","plant_count","tw_af_2020","tw_m3_2020","tw_bbl_2020"
]].to_string(index=False))

In [ ]:
total_water_demand_af = power_county_conv['tw_af_2020'].sum()
total_water_demand_m3 = power_county_conv['tw_m3_2020'].sum()
total_water_demand_bbl = power_county_conv['tw_bbl_2020'].sum()

print(f"Total Water Demand by Power Plants (2020):\n  {total_water_demand_af:.2f} acre-feet\n  {total_water_demand_m3:.2f} cubic meters\n  {total_water_demand_bbl:.2f} barrels")

### Save Converted Data

Saving the `agri_2020_conv` DataFrame to a CSV file for future use.

In [ ]:
converted_output_file_path = f'{DATA_DIR}/agri_2020_converted_units.csv'
agri_2020_conv.to_csv(converted_output_file_path, index=False)

print(f"'agri_2020_conv' DataFrame saved to: {converted_output_file_path}")

Population Projection

In [ ]:
# ── NM population data ───────────────────────────────────────────────
# From reports (Table 1) + Census QuickFacts
# census.gov/quickfacts/fact/table/NM/PST045225
POP_2015   = 2099856   # from 2015 OSE report Table 1
POP_2020   = 2117522   # from 2020 OSE report Table 1 / Census
POP_2025   = 2125498   # Census QuickFacts 2025 estimate
# Update POP_2025 from: https://www.census.gov/quickfacts/NM
# The page shows PST045225 = July 1 population estimate

pop_growth_2015_2020 = (POP_2020 - POP_2015) / POP_2015
print(f"Population growth 2015→2020: {pop_growth_2015_2020*100:.2f}%")

# Calculate annual growth rate from 2015 to 2020
years_2015_2020 = 2020 - 2015
annual_growth_rate_2015_2020 = ((POP_2020 / POP_2015)**(1/years_2015_2020)) - 1
print(f"Annual population growth rate 2015→2020: {annual_growth_rate_2015_2020*100:.2f}%")

In [ ]:
GROWTH_FACTOR = (1 + annual_growth_rate_2015_2020) ** (2025 - 2020)

# ── Apply growth to agriculture (county level) ──────────────────
agri_2025 = agri_2020_conv.copy()
for suffix in ["_af","_m3","_bbl"]:
    for prefix in ["tpwsw","tpwgw","tpw"]:
        src = f"{prefix}{suffix}"
        tgt = f"{prefix}_2025{suffix}"
        if src in agri_2025.columns:
            agri_2025[tgt] = (agri_2025[src] * GROWTH_FACTOR).round(1)

# ── Apply growth to power (county level) ───────────────
power_2025 = power_county_conv.copy()
for suffix in ["_m3_2020","_bbl_2020","_af_2020"]:
    for prefix in ["wsw","wgw","tw"]:
        src = f"{prefix}{suffix}"
        tgt = src.replace("_2020","_2025")
        if src in power_2025.columns:
            power_2025[tgt] = (power_2025[src] * GROWTH_FACTOR).round(1)

print("\nAgriculture demand — 2020 vs 2025 projection:")
print(agri_2025[["county_key","tpw_af","tpw_2025_af","tpw_2025_bbl"]]
      .sort_values("tpw_2025_bbl", ascending=False).head(10).to_string(index=False))

In [ ]:
# Merge agriculture + power into one county-level demand table
demand = (agri_2025[[
    "county_key",
    "tpwsw_af","tpwgw_af","tpw_af",           # 2020 baseline
    "tpw_2025_af","tpw_2025_m3","tpw_2025_bbl" # 2025 projection
]].rename(columns={
    "tpwsw_af":    "agri_sw_af_2020",
    "tpwgw_af":    "agri_gw_af_2020",
    "tpw_af":      "agri_tpw_af_2020",
    "tpw_2025_af": "agri_tpw_af_2025",
    "tpw_2025_m3": "agri_tpw_m3_2025",
    "tpw_2025_bbl":"agri_tpw_bbl_2025",
}).merge(
    power_2025[[
        "county_key","plant_count",
        "tw_af_2020","tw_af_2025","tw_m3_2025","tw_bbl_2025"
    ]].rename(columns={
        "tw_af_2020":  "power_tw_af_2020",
        "tw_af_2025":  "power_tw_af_2025",
        "tw_m3_2025":  "power_tw_m3_2025",
        "tw_bbl_2025": "power_tw_bbl_2025",
    }),
    on="county_key", how="left"
))

demand["plant_count"]       = demand["plant_count"].fillna(0).astype(int)
demand["power_tw_af_2025"]  = demand["power_tw_af_2025"].fillna(0)
demand["power_tw_bbl_2025"] = demand["power_tw_bbl_2025"].fillna(0)
demand["power_tw_m3_2025"]  = demand["power_tw_m3_2025"].fillna(0)

# Total demand = agri + power
demand["total_demand_af_2025"]  = (demand["agri_tpw_af_2025"] +
                                    demand["power_tw_af_2025"])
demand["total_demand_bbl_2025"] = (demand["agri_tpw_bbl_2025"] +
                                    demand["power_tw_bbl_2025"])
demand["total_demand_m3_2025"]  = (demand["agri_tpw_m3_2025"] +
                                    demand["power_tw_m3_2025"])

print("=== Final demand table (top 10 by total demand) ===")
print(demand.sort_values("total_demand_bbl_2025", ascending=False)[[
    "county_key","agri_tpw_af_2025","power_tw_af_2025",
    "total_demand_af_2025","total_demand_bbl_2025"

    ]].head(10).to_string(index=False))

In [ ]:
demand_county = (demand
    .groupby("county_key")
    .agg(
        agri_sw_af_2020  = ("agri_sw_af_2020",  "sum"),
        agri_gw_af_2020  = ("agri_gw_af_2020",  "sum"),
        agri_tpw_af_2020 = ("agri_tpw_af_2020", "sum"),
        agri_tpw_af_2025 = ("agri_tpw_af_2025", "sum"),
        agri_tpw_m3_2025 = ("agri_tpw_m3_2025", "sum"),
        agri_tpw_bbl_2025= ("agri_tpw_bbl_2025","sum"),
    )
    .reset_index()
)

In [ ]:
print(f"Before: {len(demand)} rows")
print(f"After:  {len(demand_county)} rows")   # should be ≤ 33
print(demand_county.sort_values("agri_tpw_bbl_2025", ascending=False).head(10))

In [ ]:
demand_final = demand_county.merge(
    power_2025[[
        "county_key","plant_count",
        "tw_af_2020","tw_af_2025","tw_m3_2025","tw_bbl_2025"
    ]].rename(columns={
        "tw_af_2020":  "power_tw_af_2020",
        "tw_af_2025":  "power_tw_af_2025",
        "tw_m3_2025":  "power_tw_m3_2025",
        "tw_bbl_2025": "power_tw_bbl_2025",
    }),
    on="county_key", how="left"
)
demand_final["plant_count"]        = demand_final["plant_count"].fillna(0).astype(int)
demand_final["power_tw_af_2025"]  = demand_final["power_tw_af_2025"].fillna(0)
demand_final["power_tw_bbl_2025"]  = demand_final["power_tw_bbl_2025"].fillna(0)
demand_final["power_tw_m3_2025"]  = demand_final["power_tw_m3_2025"].fillna(0)

demand_final["total_demand_af_2025"]  = (demand_final["agri_tpw_af_2025"] +
                                          demand_final["power_tw_af_2025"])
demand_final["total_demand_bbl_2025"] = (demand_final["agri_tpw_bbl_2025"] +
                                          demand_final["power_tw_bbl_2025"])
demand_final["total_demand_m3_2025"]  = (demand_final["agri_tpw_m3_2025"] +
                                          demand_final["power_tw_m3_2025"])


In [ ]:
# Remove footnote rows — any county_key that starts with a digit
# or contains long descriptive text
demand_county = demand_county[
    ~demand_county["county_key"].str.match(r"^\d")        # starts with number
    & demand_county["county_key"].str.len() < 30          # not a long footnote
].reset_index(drop=True)

print(f"Rows after cleaning: {len(demand_county)}")
print(demand_county["county_key"].tolist())

In [ ]:
# Remove footnote rows — any county_key that starts with a digit
# or contains long descriptive text
demand_county = demand_county[
    ~demand_county["county_key"].str.match(r"^\d")        # starts with number
    & demand_county["county_key"].str.len() < 30          # not a long footnote
].reset_index(drop=True)

print(f"Rows after cleaning: {len(demand_county)}")
print(demand_county["county_key"].tolist())

In [ ]:
SKIP = [
    "1_=_some_or_all", "2_=_shortage", "footnote",
    "total", "new mexico", "state total", "nm", ""
]
demand_county = demand_county[
    ~demand_county["county_key"].str.lower().isin(SKIP)
    & ~demand_county["county_key"].str.match(r"^\d")
    & (demand_county["county_key"].str.len() < 30)
].reset_index(drop=True)

In [ ]:
demand_county["county_key"].value_counts().head(10)

In [ ]:
# Save
demand.to_csv(f"{DATA_DIR}/nm_water_demand_2025_final.csv", index=False)
print("\n✓ Saved nm_water_demand_2025.csv")

In [ ]:
import pandas as pd

DEMAND_PATH = f"{WORK_DIR}/data_clean/data_processed/nm_water_demand_2025.csv"

# Load
df = pd.read_csv(DEMAND_PATH)
print(f"Before: {len(df)} rows")

# Aggregate numeric columns by county_key
numeric_cols = df.select_dtypes(include="number").columns.tolist()

df_agg = (df
    .groupby("county_key")[numeric_cols]
    .sum()
    .reset_index()
)

# Remove footnote rows
df_agg = df_agg[
    ~df_agg["county_key"].str.match(r"^\d", na=False) &
    (df_agg["county_key"].str.len() < 30)
].reset_index(drop=True)

print(f"After:  {len(df_agg)} rows")
print(df_agg["county_key"].tolist())

# Overwrite
df_agg.to_csv(DEMAND_PATH, index=False)
print("✓ Saved — one row per county")

In [ ]:
df_agg.to_csv(
    f"{WORK_DIR}/data_clean/data_processed/nm_water_demand_2025.csv",
    index=False
)
print(f"✓ Saved nm_water_demand_2025.csv — {len(df_agg)} counties")

In [ ]:
display(df_agg.describe())

In [ ]:
af_m3_columns = [col for col in df_agg.columns if '_af' in col or '_m3' in col]
df_agg_no_af_m3 = df_agg.drop(columns=af_m3_columns)
display(df_agg_no_af_m3.head())

In [ ]:
# fix : to remove _ on the county names: 02/06/2026

df_agg["county_key"] = (df_agg["county_key"]
    .str.strip()
    .str.lower()
    .str.replace("_", " ")
)

In [ ]:
output_file_path_no_af_m3 = f'{DATA_DIR}/nm_water_demand_2025_no_af_m3.csv'
df_agg_no_af_m3.to_csv(output_file_path_no_af_m3, index=False)
print(f"'df_agg_no_af_m3' DataFrame saved to: {output_file_path_no_af_m3}")